In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession

dbutils.widgets.removeAll()
dbutils.widgets.text("storage_name", "adlsproject1225")
dbutils.widgets.text("container", "data")
dbutils.widgets.text("catalog", "adventure_works")
dbutils.widgets.text("schema", "bronze")

storage_name = dbutils.widgets.get("storage_name")
container = dbutils.widgets.get("container")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

path_base = f"abfss://raw@{storage_name}.dfs.core.windows.net/"



In [0]:
customer_schema = StructType([
    StructField("CustomerID", IntegerType(), True),
    StructField("PersonID", IntegerType(), True),
    StructField("StoreID", IntegerType(), True),
    StructField("TerritoryID", IntegerType(), True),
    StructField("AccountNumber", StringType(), True),
    StructField("rowguid", StringType(), True),
    StructField("ModifiedDate", TimestampType(), True)
])

product_schema = StructType([
    StructField("ProductID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("ProductNumber", StringType(), True),
    StructField("MakeFlag", IntegerType(), True),
    StructField("FinishedGoodsFlag", IntegerType(), True),
    StructField("Color", StringType(), True),
    StructField("SafetyStockLevel", IntegerType(), True),
    StructField("ReorderPoint", IntegerType(), True),
    StructField("StandardCost", DoubleType(), True),
    StructField("ListPrice", DoubleType(), True),
    StructField("Size", StringType(), True),
    StructField("SizeUnitMeasureCode", StringType(), True),
    StructField("WeightUnitMeasureCode", StringType(), True),
    StructField("Weight", DoubleType(), True),
    StructField("DaysToManufacture", IntegerType(), True),
    StructField("ProductLine", StringType(), True),
    StructField("Class", StringType(), True),
    StructField("Style", StringType(), True),
    StructField("ProductSubcategoryID", IntegerType(), True),
    StructField("ProductModelID", IntegerType(), True),
    StructField("SellStartDate", TimestampType(), True),
    StructField("SellEndDate", TimestampType(), True),
    StructField("DiscontinuedDate", TimestampType(), True),
    StructField("rowguid", StringType(), True),
    StructField("ModifiedDate", TimestampType(), True)
])

so_detail_schema = StructType([
    StructField("SalesOrderID", IntegerType(), True),
    StructField("SalesOrderDetailID", IntegerType(), True),
    StructField("CarrierTrackingNumber", StringType(), True),
    StructField("OrderQty", IntegerType(), True),
    StructField("ProductID", IntegerType(), True),
    StructField("SpecialOfferID", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("UnitPriceDiscount", DoubleType(), True),
    StructField("LineTotal", DoubleType(), True),
    StructField("rowguid", StringType(), True),
    StructField("ModifiedDate", TimestampType(), True)
])

so_header_schema = StructType([
    StructField("SalesOrderID", IntegerType(), True),
    StructField("RevisionNumber", IntegerType(), True),
    StructField("OrderDate", TimestampType(), True),
    StructField("DueDate", TimestampType(), True),
    StructField("ShipDate", TimestampType(), True),
    StructField("Status", IntegerType(), True),
    StructField("OnlineOrderFlag", IntegerType(), True),
    StructField("SalesOrderNumber", StringType(), True),
    StructField("PurchaseOrderNumber", StringType(), True),
    StructField("AccountNumber", StringType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("SalesPersonID", IntegerType(), True),
    StructField("TerritoryID", IntegerType(), True),
    StructField("BillToAddressID", IntegerType(), True),
    StructField("ShipToAddressID", IntegerType(), True),
    StructField("ShipMethodID", IntegerType(), True),
    StructField("CreditCardID", IntegerType(), True),
    StructField("CreditCardApprovalCode", StringType(), True),
    StructField("CurrencyRateID", IntegerType(), True),
    StructField("SubTotal", DoubleType(), True),
    StructField("TaxAmt", DoubleType(), True),
    StructField("Freight", DoubleType(), True),
    StructField("TotalDue", DoubleType(), True),
    StructField("Comment", StringType(), True),
    StructField("rowguid", StringType(), True),
    StructField("ModifiedDate", TimestampType(), True)
])

territory_schema = StructType([
    StructField("TerritoryID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("CountryRegionCode", StringType(), True),
    StructField("Group", StringType(), True),
    StructField("SalesYTD", DoubleType(), True),
    StructField("SalesLastYear", DoubleType(), True),
    StructField("CostYTD", DoubleType(), True),
    StructField("CostLastYear", DoubleType(), True),
    StructField("rowguid", StringType(), True),
    StructField("ModifiedDate", TimestampType(), True)
])


In [0]:
df_customer = spark.read.option("header", True).schema(customer_schema).csv(f"{path_base}Customer.csv")
df_product = spark.read.option("header", True).schema(product_schema).csv(f"{path_base}Product.csv")
df_so_detail = spark.read.option("header", True).schema(so_detail_schema).csv(f"{path_base}SalesOrderDetail.csv")
df_so_header = spark.read.option("header", True).schema(so_header_schema).csv(f"{path_base}SalesOrderHeader.csv")
df_territory = spark.read.option("header", True).schema(territory_schema).csv(f"{path_base}SalesTerritory.csv")



In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

def add_metadata(df):
    return (
        df.withColumn("fecha_carga", current_timestamp())
          .withColumn("archivo_origen", input_file_name())
    )

df_customer = add_metadata(df_customer)
df_product = add_metadata(df_product)
df_so_detail = add_metadata(df_so_detail)
df_so_header = add_metadata(df_so_header)
df_territory = add_metadata(df_territory)

#Por experiencia se crean columnas fecha_carga y archivo_origen por auditoría, queda registrada la fecha de carga y el origen.



In [0]:
df_customer.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.customer")
df_product.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.product")
df_so_detail.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.so_detail")
df_so_header.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.so_header")
df_territory.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.territory")

